In [1]:
%load_ext autoreload
%autoreload 2

import yaml
import torch
from torch.utils.data import DataLoader
import time
import argparse
from utils.trainer_2 import load_instance, Trainer

# dtype = torch.float32
# torch.set_default_dtype(dtype)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
DEVICE = device
seed = 0
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Define available problem types and problems
PROBLEM_TYPES = ['convex', 'nonconvex', 'nonsmooth_nonconvex']
PROBLEM_NAMES = ['qp', 'qcqp', 'socp']


def create_parser(arg_list=None):
    """Create and configure the argument parser, then load and process the configuration."""
    parser = argparse.ArgumentParser(description='Neural Network Optimization')

    # General parameters
    parser.add_argument('--config', type=str, default='configs/default.yaml',
                        help='Path to YAML configuration file')
    parser.add_argument('--method', type=str,
                        help='Training method (penalty, adaptive_penalty, FSNet, DC3, projection, sup, sup_partial, sup_pen, semi, S3Net)')
    parser.add_argument('--prob_type', type=str, choices=PROBLEM_TYPES,
                        help='Problem type (convex, nonconvex, nonsmooth_nonconvex)')
    parser.add_argument('--prob_name', type=str, choices=PROBLEM_NAMES,
                        help='Problem name (qp, qcqp, socp)')
    parser.add_argument('--prob_size', type=int, nargs='+', default=[100, 50, 50, 10000],
                        help='Problem size parameters [n, m, p, N] (default: [100, 50, 50, 10000])')
    parser.add_argument('--network', type=str, default='MLP',
                        help='Type of neural network to use')
    parser.add_argument('--seed', type=int, default=2025,
                        help='Random seed for reproducibility')
    parser.add_argument('--ablation', type=bool, default=False)
    parser.add_argument('--checkpoint', type=str,
                        default=None, help='Path to model checkpoint')
    parser.add_argument('--en_subopt', type=bool, default=False,
                        help='Enable suboptimality in training')
    parser.add_argument('--subopt_ratio', type=float, default=0.0,
                        help='Suboptimality ratio if en_subopt is True')
    parser.add_argument('--save_intermediate', type=bool, default=False,
                        help='Save intermediate models during training')

    # dataset parameters
    parser.add_argument('--train_size', type=int,
                        help='Size of training dataset', default=-1)
    parser.add_argument('--batch_size', type=int,
                        help='Batch size for training')
    parser.add_argument('--val_size', type=int,
                        help='Size of validation dataset')
    parser.add_argument('--test_size', type=int, help='Size of test dataset')
    parser.add_argument('--dropout', type=float,
                        help='Dropout rate for the model')

    # Neural network parameters
    parser.add_argument('--lr', type=float, help='Learning rate')
    parser.add_argument('--lr_decay', type=float,
                        help='Learning rate decay factor')
    parser.add_argument('--lr_decay_step', type=int,
                        help='Learning rate decay step size')
    parser.add_argument('--num_epochs', type=int,
                        help='Number of training epochs')
    parser.add_argument('--hidden_dim', type=int, help='Hidden dimension size')
    parser.add_argument('--num_layers', type=int,
                        help='Number of hidden layers')

    # Feasibility seeking parameters
    parser.add_argument('--scale', type=float, help='Scale')
    parser.add_argument('--dist_weight', type=float, help='Distance weight')
    parser.add_argument('--max_diff_iter', type=int,
                        help='Maximum number of iterations for keeping the track of gradient')

    args = parser.parse_args(arg_list)

    # Load configuration from YAML file
    config_path = args.config
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Override config with command-line arguments if provided
    if args.method:
        config['seed'] = args.seed
    if args.method:
        config['method'] = args.method
    if args.prob_type:
        config['prob_type'] = args.prob_type
    if args.prob_name:
        config['prob_name'] = args.prob_name
    if args.prob_size:
        config['prob_size'] = args.prob_size
    if args.network:
        config['network'] = args.network

    config['checkpoint'] = args.checkpoint
    config['en_subopt'] = args.en_subopt
    config['subopt_ratio'] = args.subopt_ratio
    config['save_intermediate'] = args.save_intermediate

    # Override dataset parameters
    if args.batch_size:
        config['batch_size'] = args.batch_size
    if args.train_size:
        config['train_size'] = args.train_size
    if args.val_size:
        config['val_size'] = args.val_size
    if args.test_size:
        config['test_size'] = args.test_size

    # Override neural network parameters
    if args.lr:
        config['lr'] = args.lr
    if args.lr_decay:
        config['lr_decay'] = args.lr_decay
    if args.lr_decay_step:
        config['lr_decay_step'] = args.lr_decay_step
    if args.num_epochs:
        config['num_epochs'] = args.num_epochs
    if args.hidden_dim:
        config['hidden_dim'] = args.hidden_dim
    if args.num_layers:
        config['num_layers'] = args.num_layers
    if args.dropout:
        config['dropout'] = args.dropout

    # Feasibility seeking parameters
    if args.scale:
        config['FSNet']['scale'] = args.scale
        config['S3Net']['scale'] = args.scale
        config['semi']['scale'] = args.scale
    if args.dist_weight is not None:
        config['FSNet']['dist_weight'] = args.dist_weight
        config['S3Net']['dist_weight'] = args.dist_weight
        config['semi']['dist_weight'] = args.dist_weight
    if args.max_diff_iter is not None:
        config['FSNet']['max_diff_iter'] = args.max_diff_iter
        config['S3Net']['max_diff_iter'] = args.max_diff_iter
        config['semi']['max_diff_iter'] = args.max_diff_iter

    # Ablation study flag
    config['ablation'] = args.ablation

    return args, config

KeyboardInterrupt: 

In [ ]:
# (1) Supervised bootstrap initialization
args, config = create_parser([
    "--method", "FSNet",
    "--prob_type", "nonsmooth_nonconvex",
    "--prob_name", "socp",
])
opt_problem, result_save_dir = load_instance(config)
# Instantiate and use the Trainer
base_trainer = Trainer(opt_problem=opt_problem,
                       config=config, save_dir=result_save_dir)

Loading  dataset from: datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000 

Running on:  cpu


In [ ]:
test_loader = DataLoader(
    base_trainer.opt_problem.test_dataset,
    batch_size=256,
    shuffle=False
)

In [ ]:
print(test_loader)

In [ ]:
eq_weight = 1e5
ineq_weight = 1e5


def eval_merit(net, loader, use_cuda=False):
    if use_cuda:
        net = net.to(DEVICE)
    net.eval()
    val_metrics = base_trainer.evaluator.evaluate(net, loader)
    merit = val_metrics['objective'] + eq_weight * val_metrics['eq_violation_l1_mean'] + \
        ineq_weight * val_metrics['ineq_violation_l1_mean']
    return merit

In [ ]:
# (1) Supervised bootstrap initialization
args, config = create_parser([
    "--method", "FSNet",
    "--prob_type", "nonsmooth_nonconvex",
    "--prob_name", "socp",
    "--checkpoint", "results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260119-183847_MLP_FSNet_seed0_nepochs300_lr5e-05_trainsize7000_finetune_20260115-184556_sup_seedpen_model_150/model.pt",
])
opt_problem, result_save_dir = load_instance(config)
# Instantiate and use the Trainer
trainer_dummy = Trainer(opt_problem=opt_problem,
                        config=config, save_dir=result_save_dir)
model_A0 = trainer_dummy.train()  # Return model (no training)

Loading  dataset from: datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000 

Running on:  cpu
Using batch size: 512
Loading model from checkpoint: results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260119-183847_MLP_FSNet_seed0_nepochs300_lr5e-05_trainsize7000_finetune_20260115-184556_sup_seedpen_model_150/model.pt

lr: 0.0001, weight_decay: 0.001, num_epochs: 300



In [ ]:
eval_merit(model_A0, test_loader, use_cuda=False)


EVAL EVALUATION RESULTS:
Obj:     -1.008908e+00
Opt Gap:      -1.546297e+00 ± 4.125562e+00
Eq Vio l1:   5.351767e-05 (max: 1.411901e-03)
Ineq Vio l1: 7.286757e-07 (max: 7.390238e-05)
Sol Dis:   9.723220e+01 ± 2.303541e+00
Merit:             5.323744e+01 ± 1.158435e+01
Avg Inf Time:  0.3379s


np.float64(4.415726981638682)

In [ ]:
# (1) Supervised bootstrap initialization
args, config = create_parser([
    "--method", "FSNet",
    "--prob_type", "nonsmooth_nonconvex",
    "--prob_name", "socp",
    "--checkpoint", "results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260119-212322_MLP_FSNet_seed2_nepochs300_lr5e-05_trainsize7000/model.pt",
])
opt_problem, result_save_dir = load_instance(config)
# Instantiate and use the Trainer
trainer_dummy = Trainer(opt_problem=opt_problem,
                        config=config, save_dir=result_save_dir)
model_B0 = trainer_dummy.train()  # Return model (no training)

Loading  dataset from: datasets/nonsmooth_nonconvex/socp/random2025_socp_dataset_var100_ineq50_eq50_ex10000 

Running on:  cpu
Using batch size: 512
Loading model from checkpoint: results/nonsmooth_nonconvex/socp/SOCPProblem-100-50-50-10000/20260119-212322_MLP_FSNet_seed2_nepochs300_lr5e-05_trainsize7000/model.pt

lr: 0.0001, weight_decay: 0.001, num_epochs: 300



In [ ]:
eval_merit(model_B0, test_loader, use_cuda=False)


EVAL EVALUATION RESULTS:
Obj:     4.030482e+00
Opt Gap:      1.142330e+01 ± 7.823511e+00
Eq Vio l1:   3.957710e-05 (max: 5.455873e-03)
Ineq Vio l1: 3.688085e-06 (max: 9.005935e-04)
Sol Dis:   1.621014e+02 ± 2.146793e+00
Merit:             4.729567e+01 ± 3.507613e+01
Avg Inf Time:  0.3998s


np.float64(8.357000530726118)

In [ ]:
"""
    Manipulate network parameters and setup random directions with normalization.
"""

import torch
import copy

################################################################################
#                 Supporting functions for weights manipulation
################################################################################


def get_weights(net):
    """ Extract parameters from net, and return a list of tensors"""
    return [p.data for p in net.parameters()]


def set_weights(net, weights, directions=None, step=None):
    """
        Overwrite the network's weights with a specified list of tensors
        or change weights along directions with a step size.
    """
    if directions is None:
        # You cannot specify a step length without a direction.
        for (p, w) in zip(net.parameters(), weights):
            p.data.copy_(w.type(type(p.data)))
    else:
        assert step is not None, 'If a direction is specified then step must be specified as well'

        if len(directions) == 2:
            dx = directions[0]
            dy = directions[1]
            changes = [d0*step[0] + d1*step[1] for (d0, d1) in zip(dx, dy)]
        else:
            changes = [d*step for d in directions[0]]

        for (p, w, d) in zip(net.parameters(), weights, changes):
            p.data = w + torch.Tensor(d).type(type(w))


def set_states(net, states, directions=None, step=None):
    """
        Overwrite the network's state_dict or change it along directions with a step size.
    """
    if directions is None:
        net.load_state_dict(states)
    else:
        assert step is not None, 'If direction is provided then the step must be specified as well'
        if len(directions) == 2:
            dx = directions[0]
            dy = directions[1]
            changes = [d0*step[0] + d1*step[1] for (d0, d1) in zip(dx, dy)]
        else:
            changes = [d*step for d in directions[0]]

        new_states = copy.deepcopy(states)
        assert (len(new_states) == len(changes))
        for (k, v), d in zip(new_states.items(), changes):
            d = torch.tensor(d)
            v.add_(d.type(v.type()))

        net.load_state_dict(new_states)


def get_random_weights(weights):
    """
        Produce a random direction that is a list of random Gaussian tensors
        with the same shape as the network's weights, so one direction entry per weight.
    """
    return [torch.randn(w.size()) for w in weights]


def get_random_states(states):
    """
        Produce a random direction that is a list of random Gaussian tensors
        with the same shape as the network's state_dict(), so one direction entry
        per weight, including BN's running_mean/var.
    """
    return [torch.randn(w.size()) for k, w in states.items()]


def get_diff_weights(weights, weights2):
    """ Produce a direction from 'weights' to 'weights2'."""
    return [w2 - w for (w, w2) in zip(weights, weights2)]


def get_diff_states(states, states2):
    """ Produce a direction from 'states' to 'states2'."""
    return [v2 - v for (k, v), (k2, v2) in zip(states.items(), states2.items())]


################################################################################
#                        Normalization Functions
################################################################################
def normalize_direction(direction, weights, norm='filter'):
    """
        Rescale the direction so that it has similar norm as their corresponding
        model in different levels.

        Args:
          direction: a variables of the random direction for one layer
          weights: a variable of the original model for one layer
          norm: normalization method, 'filter' | 'layer' | 'weight'
    """
    if norm == 'filter':
        # Rescale the filters (weights in group) in 'direction' so that each
        # filter has the same norm as its corresponding filter in 'weights'.
        for d, w in zip(direction, weights):
            d.mul_(w.norm()/(d.norm() + 1e-10))
    elif norm == 'layer':
        # Rescale the layer variables in the direction so that each layer has
        # the same norm as the layer variables in weights.
        direction.mul_(weights.norm()/direction.norm())
    elif norm == 'weight':
        # Rescale the entries in the direction so that each entry has the same
        # scale as the corresponding weight.
        direction.mul_(weights)
    elif norm == 'dfilter':
        # Rescale the entries in the direction so that each filter direction
        # has the unit norm.
        for d in direction:
            d.div_(d.norm() + 1e-10)
    elif norm == 'dlayer':
        # Rescale the entries in the direction so that each layer direction has
        # the unit norm.
        direction.div_(direction.norm())


def normalize_directions_for_weights(direction, weights, norm='filter', ignore='biasbn'):
    """
        The normalization scales the direction entries according to the entries of weights.
    """
    assert (len(direction) == len(weights))
    for d, w in zip(direction, weights):
        if d.dim() <= 1:
            if ignore == 'biasbn':
                d.fill_(0)  # ignore directions for weights with 1 dimension
            else:
                # keep directions for weights/bias that are only 1 per node
                d.copy_(w)
        else:
            normalize_direction(d, w, norm)


def normalize_directions_for_states(direction, states, norm='filter', ignore='ignore'):
    assert (len(direction) == len(states))
    for d, (k, w) in zip(direction, states.items()):
        if d.dim() <= 1:
            if ignore == 'biasbn':
                d.fill_(0)  # ignore directions for weights with 1 dimension
            else:
                # keep directions for weights/bias that are only 1 per node
                d.copy_(w)
        else:
            normalize_direction(d, w, norm)


def ignore_biasbn(directions):
    """ Set bias and bn parameters in directions to zero """
    for d in directions:
        if d.dim() <= 1:
            d.fill_(0)


################################################################################
#                       Create directions
################################################################################
def create_target_direction(net, net2, dir_type='states'):
    """
        Setup a target direction from one model to the other

        Args:
          net: the source model
          net2: the target model with the same architecture as net.
          dir_type: 'weights' or 'states', type of directions.

        Returns:
          direction: the target direction from net to net2 with the same dimension
                     as weights or states.
    """

    assert (net2 is not None)
    # direction between net2 and net
    if dir_type == 'weights':
        w = get_weights(net)
        w2 = get_weights(net2)
        direction = get_diff_weights(w, w2)
    elif dir_type == 'states':
        s = net.state_dict()
        s2 = net2.state_dict()
        direction = get_diff_states(s, s2)

    return direction


def create_random_direction(net, dir_type='weights', ignore='biasbn', norm='filter'):
    """
        Setup a random (normalized) direction with the same dimension as
        the weights or states.

        Args:
          net: the given trained model
          dir_type: 'weights' or 'states', type of directions.
          ignore: 'biasbn', ignore biases and BN parameters.
          norm: direction normalization method, including
                'filter" | 'layer' | 'weight' | 'dlayer' | 'dfilter'

        Returns:
          direction: a random direction with the same dimension as weights or states.
    """

    # random direction
    if dir_type == 'weights':
        weights = get_weights(net)  # a list of parameters.
        direction = get_random_weights(weights)
        normalize_directions_for_weights(direction, weights, norm, ignore)
    elif dir_type == 'states':
        # a dict of parameters, including BN's running mean/var.
        states = net.state_dict()
        direction = get_random_states(states)
        normalize_directions_for_states(direction, states, norm, ignore)

    return direction

In [ ]:
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt

# ---- You already have these utilities ----
# get_weights, set_weights
# create_random_direction (or create_target_direction)


def _to_scalar(x):
    """Convert eval_merit output to a float scalar."""
    if isinstance(x, (float, int)):
        return float(x)
    if torch.is_tensor(x):
        return float(x.detach().cpu().item())
    if isinstance(x, dict):
        # if eval_merit returns dict, pick a key or define how to combine
        # Change "merit" to your actual key.
        if "merit" in x:
            return _to_scalar(x["merit"])
        # fallback: first numeric entry
        for v in x.values():
            try:
                return _to_scalar(v)
            except Exception:
                pass
        raise ValueError(
            f"Cannot convert eval_merit dict to scalar. Keys={list(x.keys())}")
    raise TypeError(f"Unsupported eval_merit return type: {type(x)}")


@torch.no_grad()
def compute_merit_surface_2d(
    model,
    test_loader,
    eval_merit,
    *,
    xmin=-1.0, xmax=1.0, xnum=51,
    ymin=-1.0, ymax=1.0, ynum=51,
    dir_type="weights",      # "weights" recommended
    ignore="biasbn",
    norm="filter",
    device=None,
    verbose=True,
):
    """
    Returns:
      X, Y: meshgrid arrays (shape [ynum, xnum])
      Z: merit array (shape [ynum, xnum]) where Z[j,i] = merit at (x_i, y_j)
      dx, dy: the sampled directions (lists of tensors)
    """
    model.eval()
    if device is not None:
        model.to(device)

    # Save base params/state
    if dir_type == "weights":
        w0 = [w.detach().clone() for w in get_weights(model)]
    elif dir_type == "states":
        s0 = copy.deepcopy(model.state_dict())
    else:
        raise ValueError("dir_type must be 'weights' or 'states'")

    # Two random directions
    dx = create_random_direction(
        model, dir_type=dir_type, ignore=ignore, norm=norm)
    dy = create_random_direction(
        model, dir_type=dir_type, ignore=ignore, norm=norm)

    xs = np.linspace(xmin, xmax, xnum, dtype=np.float32)
    ys = np.linspace(ymin, ymax, ynum, dtype=np.float32)
    Z = np.empty((ynum, xnum), dtype=np.float32)

    total = xnum * ynum
    k = 0

    for j, y in enumerate(ys):
        for i, x in enumerate(xs):
            # Move model to (x, y) along (dx, dy)
            if dir_type == "weights":
                set_weights(model, w0, directions=[
                            dx, dy], step=(float(x), float(y)))
            else:
                set_states(model, s0, directions=[
                           dx, dy], step=(float(x), float(y)))

            # Evaluate merit
            m = eval_merit(model, test_loader)
            Z[j, i] = _to_scalar(m)

            k += 1
            if verbose and (k % max(1, total // 20) == 0):
                print(
                    f"[{k:>6}/{total}] x={x:+.3f}, y={y:+.3f}, merit={Z[j, i]:.6g}")

    # Restore base params/state
    if dir_type == "weights":
        set_weights(model, w0)
    else:
        set_states(model, s0)

    X, Y = np.meshgrid(xs, ys)  # shapes [ynum, xnum]
    return X, Y, Z, dx, dy


def plot_merit_contour(
    X, Y, Z,
    *,
    title="Merit landscape",
    levels=30,
    save_path=None,   # e.g., "merit_surface.png"
    dpi=300,
    show=True,
    close=True,
):
    import matplotlib.pyplot as plt

    plt.figure()
    cs = plt.contourf(X, Y, Z, levels=levels)
    plt.colorbar(cs)
    plt.xlabel("alpha (x direction)")
    plt.ylabel("beta (y direction)")
    plt.title(title)

    if save_path is not None:
        plt.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved plot to: {save_path}")

    if show:
        plt.show()

    if close:
        plt.close()

In [ ]:
# -------------------------
# Example usage:
# -------------------------
#
X, Y, Z, dx, dy = compute_merit_surface_2d(
    model_B0,
    test_loader,
    eval_merit,
    xmin=-1, xmax=1, xnum=81,
    ymin=-1, ymax=1, ynum=81,
    dir_type="weights",
    ignore="biasbn",
    norm="filter",
    device="cuda",
)

plot_merit_contour(
    X, Y, Z,
    title="model_B0 merit landscape (2 random directions)",
    save_path="model_B0_merit_surface.png",
    dpi=300,
    show=False,   # set True if you also want it displayed
)

RuntimeError: No CUDA GPUs are available